## 06. scratchpad

In [28]:
import pickle
import pandas as pd
import Data
import re
from collections import Counter
from tqdm.notebook import tqdm

In [3]:
# with open("../data/dataframe_eng.pkl", 'rb') as f:
#     data_eng = pickle.load(f)

# with open("../data/dataframe_pol.pkl", 'rb') as f:
#     data_pol = pickle.load(f)

In [2]:
# with open("../data/bpe_tokenizer_eng.pkl", 'rb') as f:
#     tokenizer_eng = pickle.load(f)
    
# with open("../data/bpe_tokenizer_pol.pkl", 'rb') as f:
#     tokenizer_pol = pickle.load(f)

### New data

In [4]:
with open('../data/europarl/europarl-v7.pl-en.en', 'r', encoding='utf-8') as f:
    eng_lines = f.readlines()

with open('../data/europarl/europarl-v7.pl-en.pl', 'r', encoding='utf-8') as f:
    pol_lines = f.readlines()

pairs = [(e.strip(), p.strip()) for e, p in zip(eng_lines, pol_lines)
         if e.strip() and p.strip() 
         and not e.strip().startswith('<') 
         and not p.strip().startswith('<')]

df_europarl = pd.DataFrame(pairs, columns=['eng_text', 'pol_text'])
print(len(df_europarl))
df_europarl.head()

629380


,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Written statements (Rule 116): see Minutes,Oświadczenia pisemne (art. 116 Regulaminu): pa...
3,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
4,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół


In [5]:
requir = lambda x: re.sub(r"^[(]..[)]", "", x)
df_europarl['eng_text'] = df_europarl['eng_text'].apply(requir)
df_europarl['pol_text'] = df_europarl['pol_text'].apply(requir)

In [6]:
mask_eng = df_europarl['eng_text'].apply(lambda x: bool(re.search(r".+[(].+[)].+", x)))
mask_pol = df_europarl['pol_text'].apply(lambda x: bool(re.search(r".+[(].+[)].+", x)))

In [7]:
mask_eng.sum(), mask_pol.sum(), (mask_eng | mask_pol).sum()

(22461, 21196, 25231)

In [8]:
df_europarl = df_europarl[~(mask_eng | mask_pol)].reset_index(drop=True)
df_europarl

,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
3,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół
4,Membership of committees and delegations: see ...,Skład komisji i delegacji: patrz protokół
...,...,...
604144,Composition of committees and delegations : se...,Skład komisji i delegacji: Patrz protokól
604145,Agenda of the next sitting : see Minutes,Porządek obrad następnego posiedzenia: Patrz p...
604146,Closure of the sitting,Zamknięcie posiedzenia
604147,(The sitting closed at 22.25),(The sitting closed at 22.25)


In [9]:
uniq_eng = set("".join(df_europarl["eng_text"].astype(str)))
uniq_pol = set("".join(df_europarl["pol_text"].astype(str)))
print(f"Eng uniq: {len(uniq_eng)} | Pol uniq: {len(uniq_pol)}")

Eng uniq: 262 | Pol uniq: 267


In [10]:
tgt_eng = ['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ', ';']
tgt_pol = ['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ', ';']

In [11]:
eng_delchars = [x for x in sorted(list(uniq_eng), reverse=True) if x not in tgt_eng]
pol_delchars = [x for x in sorted(list(uniq_pol), reverse=True) if x not in tgt_pol]
regex_eng = "[" + "".join(map(re.escape, eng_delchars)) + "]"
regex_pol = "[" + "".join(map(re.escape, pol_delchars)) + "]"

In [13]:
mask_eng = df_europarl["eng_text"].str.contains(regex_eng, na=False)
mask_pol = df_europarl["pol_text"].str.contains(regex_pol, na=False)
mask_eng.sum(), mask_pol.sum(), (mask_eng | mask_pol).sum()

(10115, 21667, 24257)

In [14]:
df_europarl = df_europarl[~(mask_eng | mask_pol)].reset_index(drop=True)
#df_europarl = df_europarl[~mask_eng].reset_index(drop=True)

In [15]:
uniq_eng = set("".join(df_europarl["eng_text"].astype(str)))
uniq_pol = set("".join(df_europarl["pol_text"].astype(str)))
print(f"Eng uniq: {len(uniq_eng)} | Pol uniq: {len(uniq_pol)}")

Eng uniq: 78 | Pol uniq: 94


In [17]:
mask_1 = (df_europarl["eng_text"].str[-1] == '.') & (df_europarl["pol_text"].str[-1] != '.')
mask_2 = (df_europarl["eng_text"].str[-1] != '.') & (df_europarl["pol_text"].str[-1] == '.')

In [18]:
mask_1.sum(), mask_2.sum(), (mask_1 | mask_2).sum()

(1960, 1108, 3068)

In [19]:
(~(mask_1 | mask_2)).sum()

576824

In [20]:
df_europarl = df_europarl[~(mask_1 | mask_2)].reset_index(drop=True)

In [21]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [22]:
df_europarl['eng_split'] = df_europarl["eng_text"].apply(tokenize_eng)
df_europarl['pol_split'] = df_europarl["pol_text"].apply(tokenize_pol)

In [23]:
df_europarl['eng_split']

0         [action, taken, on, parliament, 's, resolution...
1                    [documents, received, :, see, minutes]
2         [texts, of, agreements, forwarded, by, the, co...
3             [membership, of, parliament, :, see, minutes]
4         [membership, of, committees, and, delegations,...
                                ...                        
576819    [composition, of, committees, and, delegations...
576820    [agenda, of, the, next, sitting, :, see, minutes]
576821                          [closure, of, the, sitting]
576822          [(, the, sitting, closed, at, 22, ., 25, )]
576823                                               [., -]
Name: eng_split, Length: 576824, dtype: object

In [24]:
uniq_freq_eng = Counter(df_europarl['eng_split'].explode())
uniq_freq_pol = Counter(df_europarl['pol_split'].explode())

In [25]:
char_factor = lambda word: [x for x in word] + ['_']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common()]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

50655 [[['t', 'h', 'e', '_'], 1019855], [[',', '_'], 712258], [['.', '_'], 562408], [['o', 'f', '_'], 488571], [['t', 'o', '_'], 458481]] 

158653 [[[',', '_'], 876176], [['.', '_'], 578767], [['w', '_'], 432544], [['i', '_'], 304762], [['n', 'a', '_'], 192303]]


In [26]:
pair_vocab_eng = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_chars_eng]
pair_vocab_pol = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_chars_pol]

In [29]:
from BPE_tokenizer import BPETokenizer

In [30]:
tokenizer_eng = BPETokenizer(vocab_chars_eng, pair_vocab_eng, 12000, 600, False)
tokenizer_eng.train_bpe()

  0%|          | 0/11945 [00:00<?, ?it/s]

In [32]:
tokenizer_pol = BPETokenizer(vocab_chars_pol, pair_vocab_pol, 24000, 600, True)
tokenizer_pol.train_bpe()

  0%|          | 0/23935 [00:00<?, ?it/s]

In [233]:
import copy

In [237]:
df_eng = df_europarl[['eng_text', 'eng_split']].copy()
df_pol = df_europarl[['pol_text', 'pol_split']].copy()

In [238]:
df_eng.head()

,eng_text,eng_split
0,Action taken on Parliament's resolutions: see ...,"[action, taken, on, parliament, 's, resolution..."
1,Documents received: see Minutes,"[documents, received, :, see, minutes]"
2,Texts of agreements forwarded by the Council: ...,"[texts, of, agreements, forwarded, by, the, co..."
3,Membership of Parliament: see Minutes,"[membership, of, parliament, :, see, minutes]"
4,Membership of committees and delegations: see ...,"[membership, of, committees, and, delegations,..."


In [239]:
from tqdm.auto import tqdm
tqdm.pandas()

In [240]:
class BPEEncoder_2():
    def __init__(self, bpe_vocab, bpe_vocab_chrs, thres_tup=55, is_tgt=False):
        self.vocab_encoder = {"".join(k): v for k, v in bpe_vocab.items()}
        self.word_check = {"".join(k): [self.vocab_encoder.get(x) for x in k] for k, _ in bpe_vocab_chrs}
        self.vocab_tuple = list(bpe_vocab.keys())[thres_tup:]
        self.char_factor = lambda word: [x for x in word] + ['_']

    def encode_snt(self, snt_toks):
        return [y for x in snt_toks for y in self.encode_word(x)]

    def encode_word(self, word):
        word_factor = self.char_factor(word)
        word_id = self.word_check.get("".join(word_factor), None)
        if word_id is not None:
            yield from word_id
        else:
            word_pairs = list(zip(word_factor, word_factor[1:]))
            for pair in self.vocab_tuple:
                if pair in word_pairs:
                    p_1, p_2 = map(re.escape, pair)
                    word_factor = re.sub(rf"(?<=\s){p_1}\s{p_2}(?=\s)", f"{pair[0]}{pair[1]}", f" {' '.join(word_factor)} ").split()
                    word_pairs = list(zip(word_factor, word_factor[1:]))
            yield from [self.vocab_encoder[x] for x in word_factor]

In [230]:
encoder_pol2 = BPEEncoder_2(tokenizer_pol.vocab, tokenizer_pol.vocab_chrs, 65, True)

In [241]:
df_pol['pol_ids'] = df_pol['pol_split'].progress_apply(encoder_pol2.encode_snt)

  0%|          | 0/576824 [00:00<?, ?it/s]

In [242]:
encoder_eng2 = BPEEncoder_2(tokenizer_eng.vocab, tokenizer_eng.vocab_chrs, 55, False)
df_eng['eng_ids'] = df_eng['eng_split'].progress_apply(encoder_eng2.encode_snt)

  0%|          | 0/576824 [00:00<?, ?it/s]

In [252]:
df_eng['eng_ids'] = df_eng['eng_ids'].apply(lambda tab: tab+[2])
df_pol['pol_ids'] = df_pol['pol_ids'].apply(lambda tab: [4]+tab+[2])

In [286]:
df_final = pd.concat([df_eng, df_pol], axis=1)[['eng_text', 'eng_ids', 'pol_text', 'pol_ids']]
df_final.sample(10)

,eng_text,eng_ids,pol_text,pol_ids
90862,In fact the territory is currently occupied by...,"[85, 700, 61, 3276, 83, 1548, 7080, 201, 7886,...","W rzeczywistości, terytorium to jest obecnie o...","[4, 78, 2508, 71, 3384, 165, 167, 893, 12521, ..."
490215,"Madam President, subsidiarity, a matter for i...","[875, 296, 68, 3767, 68, 91, 1196, 121, 1815, ...","Pani Przewodnicząca! Pomocniczość, jako spraw...","[4, 528, 1029, 376, 23174, 71, 645, 2552, 256,..."
89000,on behalf of the IND/DEM Group. - Mr President...,"[73, 1594, 78, 61, 1297, 1512, 8953, 886, 74, ...",w imieniu grupy IND/DEM. - Panie przewodnicząc...,"[4, 78, 1818, 1380, 14031, 1875, 1431, 82, 216..."
185995,This is a wholly unacceptable situation in the...,"[129, 83, 91, 8444, 2598, 674, 85, 61, 172, 28...",Taka sytuacja jest absolutnie nie do przyjęcia...,"[4, 2577, 1701, 167, 5459, 95, 144, 2149, 78, ..."
447141,"Moreover, the target of reducing carbon emissi...","[2408, 68, 61, 3012, 78, 2447, 3011, 1737, 219...",Ponadto cel w postaci obniżenia emisji dwutlen...,"[4, 1424, 2253, 78, 3940, 8430, 1985, 5475, 39..."
259708,"Europe is a united entity, and so should be th...","[346, 83, 91, 1202, 8697, 68, 81, 186, 266, 15...",Europa jest zjednoczoną całością i w ten sposó...,"[4, 1279, 167, 1494, 2289, 537, 879, 67, 78, 5..."
295101,May I give you one piece of advice: keep Europ...,"[904, 125, 1082, 305, 265, 5168, 78, 5383, 434...",Jeśli mógłbym panu coś doradzić: niech pan spr...,"[4, 757, 12633, 1692, 2621, 166, 4953, 520, 60..."
464832,We therefore ask for more streamlined procedur...,"[122, 471, 1069, 121, 300, 5684, 4094, 1995, 1...",Dlatego prosimy o usprawnienie procedur dla do...,"[4, 596, 9947, 68, 11545, 3457, 256, 3915, 496..."
476800,I also wish to say how pleased I am that the a...,"[125, 238, 1446, 80, 738, 713, 1899, 125, 354,...","Chciałbym również powiedzieć, jak bardzo ciesz...","[4, 708, 391, 1141, 71, 268, 467, 2697, 140, 7..."
445469,"In this regard, the Ombudsman tends readily to...","[85, 129, 1035, 68, 61, 3521, 11313, 647, 1526...",Pod tym względem rzecznik chętnie wydaje oświa...,"[4, 792, 266, 1983, 9823, 9935, 2143, 3229, 25..."


In [287]:
with open("../data/dataframe_europarl.pkl", 'wb') as f:
    pickle.dump(df_final, f)

In [288]:
with open("../data/tokenizer_euro_pol.pkl", 'wb') as f:
    pickle.dump(tokenizer_pol, f)

with open("../data/tokenizer_euro_eng.pkl", 'wb') as f:
    pickle.dump(tokenizer_eng, f)